# Online Retail II - Data Cleaning, SQL Database & KPI Summary

This is the first notebook in an end-to-end data analyst project built on the **Online Retail II** dataset - real transaction data from a UK-based online retailer, covering December 2009 to December 2011 (roughly 1 million rows).

**What this notebook covers:**
1. Cleaning the raw Excel data in Python
2. Loading the cleaned data into a SQLite database using a simple star schema
3. Generating a small Excel file with a few KPI tables (using live formulas, not hardcoded numbers)

The rest of the project - the advanced SQL queries (RFM, cohort analysis, etc.) and the Power BI dashboard - builds on top of what gets created here, so getting this part right matters for everything downstream.

**Tools used:** Python (pandas), SQLite, openpyxl.

The paths below are set up for my own folder:
`E:/Education/Data Analysis/Project/Online Retail Shop/`

If you've kept the project somewhere else, just change the `BASE_DIR` line below - everything else is built from that automatically.

## Step 0: Setup - base folder and the sub-folders we need

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import matplotlib.pyplot as plt

# Change this one line if your project folder is somewhere else.
# Everything else below is built automatically from this.
BASE_DIR = "E:/Education/Data Analysis/Project/Online Retail Shop"

DATA_DIR = f"{BASE_DIR}/data"
REPORT_DIR = f"{BASE_DIR}/report"

RAW_FILE = f"{DATA_DIR}/online_retail_II.xlsx"
CLEANED_CSV = f"{DATA_DIR}/cleaned_retail_data.csv"
REMOVED_LOG_CSV = f"{DATA_DIR}/removed_rows_summary.csv"
DB_PATH = f"{DATA_DIR}/online_retail.db"
KPI_XLSX = f"{REPORT_DIR}/KPI_Summary.xlsx"

# make sure the folders exist so saving/writing never fails
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

pd.set_option("display.max_columns", None)

print("Raw file expected at:", RAW_FILE)
print("Raw file exists? ->", os.path.exists(RAW_FILE))


ModuleNotFoundError: No module named 'pandas'

If the line above prints `False`, that means `online_retail_II.xlsx` isn't in the `data` folder yet. Copy it there first, then keep going.

## Step 1: Load the raw data

The dataset is split across two sheets, `Year 2009-2010` and `Year 2010-2011`. I'm loading both and combining them into one DataFrame so every cleaning step below only has to be written once.

In [ ]:
print("Loading Excel sheets... (this can take 30-60 seconds for 1M+ rows)")

sheet_2009_2010 = pd.read_excel(RAW_FILE, sheet_name="Year 2009-2010")
sheet_2010_2011 = pd.read_excel(RAW_FILE, sheet_name="Year 2010-2011")

df = pd.concat([sheet_2009_2010, sheet_2010_2011], ignore_index=True)
print(f"Loaded {len(df):,} raw rows from both sheets.")
df.head()


## Step 2: Cleaning the data

Real transaction data is messy - duplicate rows, missing descriptions, cancelled orders, test entries that aren't actual products, and so on. I kept each cleaning decision in its own cell instead of one giant block, so it's easy to see exactly what got removed and why. The `removal_log` list tracks the row count for every step, which gets turned into a summary table at the end (Step 2.9).

### 2.1 - Rename columns to snake_case

In [ ]:
df = df.rename(columns={
    "Invoice": "invoice_no",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country",
})

removal_log = []
start_count = len(df)


### 2.2 - Fix data types

In [ ]:
df["invoice_no"] = df["invoice_no"].astype(str).str.strip()
df["stock_code"] = df["stock_code"].astype(str).str.strip()
df["description"] = df["description"].astype(str).str.strip()
df["country"] = df["country"].astype(str).str.strip()
df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")
df.dtypes


### 2.3 - Drop exact duplicate rows

In [ ]:
before = len(df)
df = df.drop_duplicates()
removal_log.append(("exact_duplicate_rows", before - len(df)))
print(f"Removed {before - len(df):,} exact duplicate rows")


### 2.4 - Handle missing or blank descriptions

In [ ]:
before = len(df)
df = df[df["description"].notna() & (df["description"].str.strip() != "") & (df["description"].str.lower() != "nan")]
removal_log.append(("missing_or_blank_description", before - len(df)))
print(f"Removed {before - len(df):,} rows with missing/blank description")


### 2.5 - Flag cancelled orders

Invoice numbers starting with `C` are cancellations/returns, not actual sales. I'm not deleting these rows - just flagging them with `is_cancelled` - so I can exclude them from revenue and product numbers later without losing the information that a cancellation happened at all.

In [ ]:
df["is_cancelled"] = df["invoice_no"].str.startswith("C")
print(df["is_cancelled"].value_counts())


### 2.6 - Remove non-product stock codes

Codes like postage, bank charges, and test entries aren't real products. Left in, they'd show up in "top products" and skew the numbers, so they're dropped here.

In [ ]:
non_product_codes = ["POST", "D", "M", "DOT", "BANK CHARGES", "AMAZONFEE",
                      "PADS", "CRUK", "C2", "S", "TEST001", "TEST002"]
before = len(df)
df = df[~df["stock_code"].str.upper().isin(non_product_codes)]
removal_log.append(("non_product_stock_codes", before - len(df)))
print(f"Removed {before - len(df):,} non-product rows")


### 2.7 - Remove zero/negative price rows, zero-quantity rows, and fix the Customer ID type

A price of zero or less usually means a data entry error or an internal adjustment rather than a real sale, and the same goes for zero quantity. I'm also converting Customer ID to a nullable integer type, since a fair number of rows don't have one - most likely guest checkouts.

In [ ]:
before = len(df)
df = df[df["unit_price"] > 0]
removal_log.append(("zero_or_negative_unit_price", before - len(df)))

before = len(df)
df = df[df["quantity"] != 0]
removal_log.append(("zero_quantity", before - len(df)))

df["has_customer_id"] = df["customer_id"].notna()
df["customer_id"] = df["customer_id"].astype("Int64")

before = len(df)
df = df[df["invoice_date"].notna()]
removal_log.append(("missing_invoice_date", before - len(df)))

print("Cleaning steps done.")


### 2.8 - Add a few derived columns

`line_revenue` is the one used everywhere from here on (charts, SQL, Power BI), and `invoice_year_month` makes monthly grouping a one-liner instead of extracting year and month separately every time.

In [ ]:
df["line_revenue"] = df["quantity"] * df["unit_price"]
df["invoice_year"] = df["invoice_date"].dt.year
df["invoice_month"] = df["invoice_date"].dt.month
# invoice_year_month gives something like "2010-05" for each row,
# which makes grouping by month a lot easier later on.
df["invoice_year_month"] = df["invoice_date"].dt.strftime("%Y-%m")
df = df.reset_index(drop=True)
df.head()


### 2.9 - Cleaning summary

In [ ]:
log_df = pd.DataFrame(removal_log, columns=["reason", "rows_removed"])
log_df.loc[len(log_df)] = ["TOTAL_STARTING_ROWS", start_count]
log_df.loc[len(log_df)] = ["TOTAL_FINAL_ROWS", len(df)]
log_df.to_csv(REMOVED_LOG_CSV, index=False)

print(f"Final shape: {df.shape}")
log_df


## Step 3: A few quick charts to sanity-check the data

Before this goes anywhere near SQL or Power BI, it's worth eyeballing it with a couple of quick charts. If something looks obviously off here, it's a lot cheaper to catch it now than after the whole pipeline is built on top of it.

### 3.1 - Monthly revenue trend

In [ ]:
monthly_revenue = (
    df[df["is_cancelled"] == False]
    .groupby("invoice_year_month")["line_revenue"]
    .sum()
    .sort_index()
)

plt.figure(figsize=(12, 4))
monthly_revenue.plot(kind="line", marker="o")
plt.title("Monthly Revenue Trend")
plt.ylabel("Revenue (GBP)")
plt.xlabel("Year-Month")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 3.2 - Top 10 products by revenue

In [ ]:
top_products_chart = (
    df[df["is_cancelled"] == False]
    .groupby("description")["line_revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 5))
top_products_chart.sort_values().plot(kind="barh")
plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue (GBP)")
plt.tight_layout()
plt.show()


### 3.3 - Revenue by country (top 10, log scale)

In [ ]:
top_countries_chart = (
    df[df["is_cancelled"] == False]
    .groupby("country")["line_revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 5))
top_countries_chart.sort_values().plot(kind="barh", logx=True)
plt.title("Top 10 Countries by Revenue (log scale)")
plt.xlabel("Revenue (GBP, log scale)")
plt.tight_layout()
plt.show()


## Step 4: Save the cleaned data to CSV

This CSV is the single source of truth from here on - it's what gets loaded into SQLite in the next step, and Power BI can also read it directly for anything that doesn't need to go through SQL.

In [ ]:
df.to_csv(CLEANED_CSV, index=False)
print(f"Saved cleaned data -> {CLEANED_CSV}")
print(f"Final shape: {df.shape}")


## Step 5: Load into SQLite (star schema)

Three tables - `dim_customers`, `dim_products`, and `fact_sales` - with foreign keys and indexes. I went with a small star schema instead of one flat table mainly because it's what the later SQL work (RFM, cohort/retention, joins across products and customers) is actually written against, and it's closer to what a real reporting database would look like.

### 5.1 - Create the schema

In [ ]:
SCHEMA_SQL = """
DROP TABLE IF EXISTS fact_sales;
DROP TABLE IF EXISTS dim_products;
DROP TABLE IF EXISTS dim_customers;

CREATE TABLE dim_customers (
    customer_id     INTEGER PRIMARY KEY,
    country         TEXT
);

CREATE TABLE dim_products (
    stock_code      TEXT PRIMARY KEY,
    description     TEXT
);

CREATE TABLE fact_sales (
    sale_id             INTEGER PRIMARY KEY AUTOINCREMENT,
    invoice_no          TEXT NOT NULL,
    stock_code          TEXT NOT NULL,
    customer_id         INTEGER,
    invoice_date        TEXT NOT NULL,
    quantity            INTEGER NOT NULL,
    unit_price          REAL NOT NULL,
    line_revenue        REAL NOT NULL,
    country              TEXT,
    is_cancelled        INTEGER NOT NULL,
    has_customer_id     INTEGER NOT NULL,
    invoice_year_month  TEXT NOT NULL,
    FOREIGN KEY (stock_code) REFERENCES dim_products(stock_code),
    FOREIGN KEY (customer_id) REFERENCES dim_customers(customer_id)
);

CREATE INDEX idx_fact_sales_customer  ON fact_sales(customer_id);
CREATE INDEX idx_fact_sales_stock     ON fact_sales(stock_code);
CREATE INDEX idx_fact_sales_date      ON fact_sales(invoice_date);
CREATE INDEX idx_fact_sales_yearmonth ON fact_sales(invoice_year_month);
"""

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.executescript(SCHEMA_SQL)
conn.commit()
print("Schema created.")


### 5.2 - Build dim_customers

In [ ]:
# a customer can occasionally show more than one country (data entry
# mistakes happen), so I just keep the first country I see for each
# customer - good enough for this project
known_customers = df[df["has_customer_id"]]
customers_df = known_customers.drop_duplicates(subset="customer_id", keep="first")
customers_df = customers_df[["customer_id", "country"]]
customers_df.to_sql("dim_customers", conn, if_exists="append", index=False)
print(f"dim_customers: {len(customers_df):,} rows")
customers_df.head()


### 5.3 - Build dim_products

In [ ]:
# same idea here - if a stock_code has slightly different descriptions
# across rows, I just keep the first one
products_df = df.drop_duplicates(subset="stock_code", keep="first")
products_df = products_df[["stock_code", "description"]]
products_df.to_sql("dim_products", conn, if_exists="append", index=False)
print(f"dim_products: {len(products_df):,} rows")
products_df.head()


### 5.4 - Build fact_sales

In [ ]:
fact_df = df[[
    "invoice_no", "stock_code", "customer_id", "invoice_date",
    "quantity", "unit_price", "line_revenue", "country",
    "is_cancelled", "has_customer_id", "invoice_year_month",
]].copy()
fact_df["invoice_date"] = fact_df["invoice_date"].astype(str)
fact_df["is_cancelled"] = fact_df["is_cancelled"].astype(int)
fact_df["has_customer_id"] = fact_df["has_customer_id"].astype(int)
fact_df.to_sql("fact_sales", conn, if_exists="append", index=False)
conn.commit()
print(f"fact_sales: {len(fact_df):,} rows")


### 5.5 - Quick sanity check

In [ ]:
for table in ["dim_customers", "dim_products", "fact_sales"]:
    count = cur.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count:,} rows")


## Step 6: Build a small KPI Excel summary

This isn't meant to replace the Power BI dashboard - it's a lightweight, shareable summary that anyone can open in Excel without needing a BI tool installed. I'm not dumping the raw data in here, just a few small aggregated tables, plus a KPI Dashboard sheet that uses live formulas instead of hardcoded numbers so the numbers stay correct if the underlying tables ever change.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

HEADER_FILL = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
HEADER_FONT = Font(name="Arial", bold=True, color="FFFFFF")
TITLE_FONT = Font(name="Arial", bold=True, size=14, color="1F4E78")
BODY_FONT = Font(name="Arial", size=10)

def write_dataframe(ws, dframe, start_row=1):
    for col_idx, col_name in enumerate(dframe.columns, start=1):
        cell = ws.cell(row=start_row, column=col_idx, value=col_name)
        cell.font = HEADER_FONT
        cell.fill = HEADER_FILL
        cell.alignment = Alignment(horizontal="center")
    for row_idx, row in enumerate(dframe.itertuples(index=False), start=start_row + 1):
        for col_idx, value in enumerate(row, start=1):
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.font = BODY_FONT
    # widen columns so the text is actually readable
    for col_idx, col_name in enumerate(dframe.columns, start=1):
        longest_value = len(str(col_name))
        for value in dframe[col_name].head(50):
            if len(str(value)) > longest_value:
                longest_value = len(str(value))
        column_letter = get_column_letter(col_idx)
        ws.column_dimensions[column_letter].width = min(longest_value + 3, 40)
    return start_row + len(dframe) + 1


In [ ]:
monthly_kpi = pd.read_sql("""
    SELECT invoice_year_month, ROUND(SUM(line_revenue),2) AS total_revenue,
           COUNT(DISTINCT invoice_no) AS total_orders
    FROM fact_sales WHERE is_cancelled = 0
    GROUP BY invoice_year_month ORDER BY invoice_year_month
""", conn)

country_kpi = pd.read_sql("""
    SELECT country, ROUND(SUM(line_revenue),2) AS total_revenue,
           COUNT(DISTINCT invoice_no) AS total_orders,
           COUNT(DISTINCT customer_id) AS unique_customers
    FROM fact_sales WHERE is_cancelled = 0
    GROUP BY country ORDER BY total_revenue DESC LIMIT 15
""", conn)

products_kpi = pd.read_sql("""
    SELECT fs.stock_code, dp.description,
           SUM(fs.quantity) AS units_sold,
           ROUND(SUM(fs.line_revenue),2) AS total_revenue
    FROM fact_sales fs JOIN dim_products dp ON fs.stock_code = dp.stock_code
    WHERE fs.is_cancelled = 0
    GROUP BY fs.stock_code, dp.description
    ORDER BY total_revenue DESC LIMIT 15
""", conn)

conn.close()
print("KPI source tables ready.")


In [ ]:
wb = Workbook()

ws_monthly = wb.active
ws_monthly.title = "Monthly Revenue"
write_dataframe(ws_monthly, monthly_kpi)
n_month_rows = len(monthly_kpi)

ws_country = wb.create_sheet("Country Sales")
write_dataframe(ws_country, country_kpi)
n_country_rows = len(country_kpi)

ws_products = wb.create_sheet("Top Products")
write_dataframe(ws_products, products_kpi)
n_product_rows = len(products_kpi)

ws_kpi = wb.create_sheet("KPI Dashboard", 0)
ws_kpi["A1"] = "Online Retail II - KPI Summary"
ws_kpi["A1"].font = TITLE_FONT
ws_kpi.merge_cells("A1:C1")

kpi_rows = [
    ("Total Revenue (GBP)", f"=SUM('Monthly Revenue'!B2:B{n_month_rows + 1})"),
    ("Total Orders",        f"=SUM('Monthly Revenue'!C2:C{n_month_rows + 1})"),
    ("Avg Monthly Revenue (GBP)", f"=AVERAGE('Monthly Revenue'!B2:B{n_month_rows + 1})"),
    ("Avg Order Value (GBP)", "=B4/B5"),
    ("Number of Countries", f"=COUNTA('Country Sales'!A2:A{n_country_rows + 1})"),
    ("Top Country by Revenue", "=INDEX('Country Sales'!A2:A16, MATCH(MAX('Country Sales'!B2:B16), 'Country Sales'!B2:B16, 0))"),
    ("Top Product by Revenue", "=INDEX('Top Products'!B2:B16, MATCH(MAX('Top Products'!D2:D16), 'Top Products'!D2:D16, 0))"),
]

ws_kpi["A3"] = "Metric"
ws_kpi["B3"] = "Value"
ws_kpi["A3"].font = HEADER_FONT
ws_kpi["B3"].font = HEADER_FONT
ws_kpi["A3"].fill = HEADER_FILL
ws_kpi["B3"].fill = HEADER_FILL

for i, (label, formula) in enumerate(kpi_rows, start=4):
    ws_kpi[f"A{i}"] = label
    ws_kpi[f"B{i}"] = formula
    ws_kpi[f"A{i}"].font = BODY_FONT
    ws_kpi[f"B{i}"].font = BODY_FONT

ws_kpi["B4"].number_format = "#,##0.00"
ws_kpi["B6"].number_format = "#,##0.00"
ws_kpi["B7"].number_format = "#,##0.00"

ws_kpi.column_dimensions["A"].width = 28
ws_kpi.column_dimensions["B"].width = 30
ws_kpi["A12"] = "All KPI values above are live formulas pulling from the data sheets, not hardcoded numbers."
ws_kpi["A12"].font = Font(name="Arial", size=9, italic=True, color="808080")

wb.save(KPI_XLSX)
print(f"Saved -> {KPI_XLSX}")
print("Formulas might show as text/0 until you open the file in Excel - they recalculate automatically on open.")


## Done

Once this notebook finishes running, the `data` folder will have:
- `cleaned_retail_data.csv`
- `removed_rows_summary.csv`
- `online_retail.db`

And the `report` folder will have `KPI_Summary.xlsx`.

This notebook covers the data engineering side of the project - getting messy raw data into a clean, query-ready database. The actual analysis and business insights come next.

What's left after this:
1. Run the 4 `.sql` files in the `sql/` folder against `online_retail.db` (RFM, top products, country-wise sales, cohort/retention).
2. Follow `powerbi/PowerBI_Build_Guide.md` to build the dashboard.
3. Final insights are written up in `report/Insights_and_Recommendations.md`.